# 그룹–멤버 계층 지수(Member Mention Index · MCI) — 최종 코퍼스 10,020건 검증 노트북

r22 노트북(`archive/v6_r22_era/Group_Member Pilot/group_member_fpu_pilot.ipynb`)은 23개 그룹 파일럿을 다뤘다. 최종 데이터는

| 입력 | 내용 |
|---|---|
| `member_mention_index_v7.json` | 45개 그룹, 최종 코퍼스 10,020건 기준 멤버명 언급 수·Impact Share·MCI |
| `member_mention_pilot_v6.json` | 동결 시점 23개 그룹 파일럿(7,350건 기준) — 비교용 |
| `member_pilot_mci_correlation_v7.json` | MCI ↔ 성과지표 상관·회귀 분석(v7-55 시점 8,981건 MCI × 동결 outcome) |
| `fandom_scores_live_reference_v7.json`, `fandom_scores_v6.json` | Coverage Index 가중식 검증(라이브·동결) |

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72
ARCHIVE_DIR = REPO / "archive" / "v6_r22_era" / "data" / "v6_r22_snapshot"   # r22(5,612건) 비교용, 읽기 전용


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

mm = load_json(DATA_DIR / "member_mention_index_v7.json")
pilot23 = load_json(DATA_DIR / "member_mention_pilot_v6.json")
mci_corr = load_json(DATA_DIR / "member_pilot_mci_correlation_v7.json")
live_scores = load_json(DATA_DIR / "fandom_scores_live_reference_v7.json")
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")
fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
per_fandom_corpus = {f["fandom"]: len(f.get("loyalty", [])) + len(f.get("spillover", [])) for f in fandoms}
print(f"member_mention_index_v7.json: {len(mm)}개 그룹 | pilot v6: {len(pilot23)}개 | 상관분석 그룹 수: {mci_corr['n_groups']}")
print("index 그룹 == 상관분석 그룹:", set(mm) == set(mci_corr["groups"]), "| 23개 파일럿 ⊂ 45개 index:", set(pilot23) <= set(mm))

member_mention_index_v7.json: 45개 그룹 | pilot v6: 23개 | 상관분석 그룹 수: 45
index 그룹 == 상관분석 그룹: True | 23개 파일럿 ⊂ 45개 index: True


## 1. Coverage Index 가중식 검증 (0.30 언어 + 0.25 시장 + 0.20 출처유형 + 0.15 시간 + 0.10 개체) — 라이브·동결

In [2]:
DOC_WEIGHTS = {"language": 0.30, "market": 0.25, "source_type": 0.20, "time": 0.15, "entity": 0.10}
def check_cov(scores, label):
    w_mis = re_mis = 0
    for rec in scores:
        cd = rec["coverage_detail"]
        if cd["weights"] != DOC_WEIGHTS: w_mis += 1
        rec_v = sum(cd["weights"][k] * cd[f"{k}_coverage"] for k in DOC_WEIGHTS)
        if abs(rec_v - rec["coverage_index"]) > 0.001: re_mis += 1
    print(f"[{label}] 가중치 != 문서값: {w_mis}/{len(scores)} | coverage_index 재계산 불일치: {re_mis}/{len(scores)}")
check_cov(live_scores, "라이브 10,020건"); check_cov(frozen_scores, "동결 7,350건")

[라이브 10,020건] 가중치 != 문서값: 0/100 | coverage_index 재계산 불일치: 0/100
[동결 7,350건] 가중치 != 문서값: 0/100 | coverage_index 재계산 불일치: 0/100


## 2. Member Impact Share / MCI — 45개 그룹 재계산

In [3]:
rows = []; mism = 0; tgb_mis = 0
for g, rec in mm.items():
    counts = rec["member_mention_counts"]; total = sum(counts.values())
    share = {m: (c / total if total else 0.0) for m, c in counts.items()}
    mci = sum(s ** 2 for s in share.values())
    if max(abs(share[m] - rec["member_impact_share_index"][m]) for m in counts) > 0.0015 or abs(mci - rec["mci_index"]) > 0.0015: mism += 1  # 소수 3자리 반올림 허용
    if total != rec["total_member_mentions"]: mism += 1
    if per_fandom_corpus.get(g) != rec["total_group_bullets"]: tgb_mis += 1
    rows.append({"group": g, "n_members": len(counts), "total_group_bullets": rec["total_group_bullets"], "total_mentions": total,
                 "mci_index": rec["mci_index"], "mci_recomputed": round(mci, 4), "mci_floor(1/n)": round(1 / len(counts), 4),
                 "mci_excess": round(rec["mci_index"] - 1 / len(counts), 4),
                 "top_member": max(counts, key=counts.get) if total else None, "top_member_share": round(max(share.values()), 4) if total else 0.0})
member_df = pd.DataFrame(rows).sort_values("mci_index", ascending=False).reset_index(drop=True)
print(f"Impact Share/MCI/total_member_mentions 재계산 불일치(반올림 0.0015 초과) 그룹: {mism} / {len(mm)} | total_group_bullets != 코퍼스: {tgb_mis} / {len(mm)}")
print(f"멤버 언급 총합: {member_df['total_mentions'].sum()}건 | MCI 평균 {member_df['mci_index'].mean():.4f}, 최소 {member_df['mci_index'].min()} ({member_df.loc[member_df['mci_index'].idxmin(), 'group']}), 최대 {member_df['mci_index'].max()} ({member_df.loc[member_df['mci_index'].idxmax(), 'group']})")
print("상관분석 JSON 기술통계(v7-55 시점):", mci_corr["mci_descriptives"])
member_df

Impact Share/MCI/total_member_mentions 재계산 불일치(반올림 0.0015 초과) 그룹: 0 / 45 | total_group_bullets != 코퍼스: 0 / 45
멤버 언급 총합: 1730건 | MCI 평균 0.2606, 최소 0.111 (NCT), 최대 0.66 (FTISLAND)
상관분석 JSON 기술통계(v7-55 시점): {'mean': 0.2662, 'sd': 0.1096, 'min': 0.111, 'max': 0.649, 'min_group': 'NCT', 'max_group': 'FTISLAND'}


,group,n_members,total_group_bullets,total_mentions,mci_index,mci_recomputed,mci_floor(1/n),mci_excess,top_member,top_member_share
0,FTISLAND,2,73,23,0.660,0.6597,0.5000,0.1600,이홍기,0.7826
1,동방신기,2,82,33,0.512,0.5115,0.5000,0.0120,유노윤호,0.5758
2,CNBLUE,3,84,24,0.504,0.5035,0.3333,0.1707,정용화,0.6667
3,CORTIS,5,135,23,0.459,0.4594,0.2000,0.2590,제임스,0.6522
4,빅뱅,4,97,32,0.364,0.3633,0.2500,0.1140,지드래곤,0.4688
5,RIIZE,6,121,26,0.358,0.3580,0.1667,0.1913,쇼타로,0.5385
6,GOT7,7,107,53,0.316,0.3158,0.1429,0.1731,뱀뱀,0.4717
7,잭스키스,4,85,19,0.307,0.3075,0.2500,0.0570,은지원,0.3684
8,BLACKPINK,4,195,63,0.301,0.3016,0.2500,0.0510,리사,0.4444
9,Stray Kids,8,163,44,0.292,0.2924,0.1250,0.1670,필릭스,0.4091


### 2-1. 빅뱅(BIGBANG) — 전략문서가 직접 예시로 든 그룹 (r22: 탑 0건 → 최종: 탑 1건)

In [4]:
bb = mm["빅뱅"]; bb23 = pilot23["빅뱅"]
bb_detail = pd.DataFrame([{"member": m, "mentions(최종)": bb["member_mention_counts"][m], "impact_share(최종)": bb["member_impact_share_index"][m],
                           "mentions(동결 파일럿)": bb23["member_mention_counts"].get(m), "impact_share(동결 파일럿)": bb23["member_impact_share_pilot"].get(m)}
                          for m in bb["member_mention_counts"]]).sort_values("impact_share(최종)", ascending=False).reset_index(drop=True)
print(f"빅뱅 total_group_bullets {bb['total_group_bullets']} (동결 {bb23['total_group_bullets']}) | 멤버 언급 {bb['total_member_mentions']} (동결 {bb23['total_member_mentions']}) | MCI {bb['mci_index']} (동결 {bb23['mci_pilot']})")
print("문서 예시 4명(지드래곤/태양/대성/탑)과 일치:", set(bb["member_mention_counts"]) == {"지드래곤", "태양", "대성", "탑"})
bb_detail

빅뱅 total_group_bullets 97 (동결 72) | 멤버 언급 32 (동결 24) | MCI 0.364 (동결 0.389)
문서 예시 4명(지드래곤/태양/대성/탑)과 일치: True


,member,mentions(최종),impact_share(최종),mentions(동결 파일럿),impact_share(동결 파일럿)
0,지드래곤,15,0.469,12,0.500
1,태양,11,0.344,8,0.333
2,대성,5,0.156,4,0.167
3,탑,1,0.031,0,0.000


## 3. 동결 23개 파일럿 → 최종 45개 지수: 같은 그룹의 변화

In [5]:
chg = pd.DataFrame([{"group": g, "bullets(동결)": pilot23[g]["total_group_bullets"], "bullets(최종)": mm[g]["total_group_bullets"],
                     "mentions(동결)": pilot23[g]["total_member_mentions"], "mentions(최종)": mm[g]["total_member_mentions"],
                     "MCI(동결)": pilot23[g]["mci_pilot"], "MCI(최종)": mm[g]["mci_index"]} for g in pilot23])
chg["ΔMCI"] = (chg["MCI(최종)"] - chg["MCI(동결)"]).round(3)
print("최종에서 새로 추가된 22개 그룹:", sorted(set(mm) - set(pilot23)))
print(f"23개 그룹 MCI 변화: 평균 Δ {chg['ΔMCI'].mean():+.4f}, |Δ|>0.05 인 그룹 {(chg['ΔMCI'].abs() > 0.05).sum()}개")
chg.sort_values("ΔMCI", key=lambda s: s.abs(), ascending=False)

최종에서 새로 추가된 22개 그룹: ['A-pink', 'ATEEZ', 'BTOB', 'CNBLUE', 'DAY6', 'FTISLAND', 'GOT7', 'HOT', 'IKON', 'ITZY', 'QWER', 'STAYC', 'god', '동방신기', '레드벨벳', '리센느(RESCENE)', '아이오아이', '여자친구', '오마이걸', '워너원', '잭스키스', '프로미스나인']
23개 그룹 MCI 변화: 평균 Δ -0.0903, |Δ|>0.05 인 그룹 7개


,group,bullets(동결),bullets(최종),mentions(동결),mentions(최종),MCI(동결),MCI(최종),ΔMCI
21,Hearts2Hearts,70,102,9,34,1.000,0.289,-0.711
22,CORTIS,111,135,7,23,1.000,0.459,-0.541
17,RIIZE,100,121,13,26,0.621,0.358,-0.263
20,엔믹스,93,110,11,24,0.323,0.233,-0.090
19,NewJeans,122,154,27,41,0.346,0.273,-0.073
9,IVE,70,100,15,29,0.307,0.239,-0.068
16,LE SSERAFIM,109,143,20,33,0.340,0.278,-0.062
13,Stray Kids,138,163,34,44,0.341,0.292,-0.049
7,(여자)아이들,83,118,24,48,0.302,0.256,-0.046
2,BLACKPINK,146,195,39,63,0.344,0.301,-0.043


## 4. MCI ↔ 멤버 수·성과지표 상관 — 상관분석 JSON 값의 재현 (JSON은 v7-55 시점 MCI, 여기서는 최종 MCI로 재계산해 차이를 본다)

In [6]:
from scipy import stats
frozen_by = {d["fandom"]: d for d in frozen_scores}
sub = member_df[member_df["group"].isin(frozen_by)].copy()
for col in ["loyalty_score", "spillover_score", "coverage_index", "factor_diversity"]:
    sub[col] = sub["group"].map({g: frozen_by[g][col] for g in frozen_by})
r_n, p_n = stats.pearsonr(sub["mci_index"], sub["n_members"])
print(f"n={len(sub)} (동결 로스터에 있는 그룹) | r(MCI, 멤버 수) = {r_n:.4f} (p={p_n:.2e})  ← JSON: {mci_corr['mci_vs_member_count']['pearson_r']}")
rows = []
for col in ["loyalty_score", "spillover_score", "coverage_index", "factor_diversity"]:
    r_raw, p_raw = stats.pearsonr(sub["mci_index"], sub[col]); r_ex, p_ex = stats.pearsonr(sub["mci_excess"], sub[col])
    rows.append({"outcome(동결)": col, "r(MCI 최종)": round(r_raw, 4), "p": round(p_raw, 4), "JSON r(MCI v7-55)": mci_corr["correlations_mci_raw"][col]["pearson_r"],
                 "r(MCI_excess 최종)": round(r_ex, 4), "JSON r(MCI_excess)": mci_corr["correlations_mci_excess"][col]["pearson_r"]})
print("verdict(JSON):", mci_corr["verdict"]["summary_ko"][:200], "...")
print("caveat:", mci_corr["caveat_temporal_mismatch"][:150], "...")
pd.DataFrame(rows)

n=45 (동결 로스터에 있는 그룹) | r(MCI, 멤버 수) = -0.7280 (p=1.46e-08)  ← JSON: -0.7489
verdict(JSON): 원시 MCI가 4개 outcome 지표 중 가장 잘 설명하는 것은 loyalty_score로 R²=0.1546이지만, 멤버 수를 통제한 MCI_excess 기준으로는 spillover_score에 대해 R²=0.0357로 낮아진다. MCI와 member_count의 원 상관은 r=-0.7489로, 원시 MCI의 설명력 상당 부분이 멤버 수 차이에서 비롯된  ...
caveat: MCI는 v7 55 시점 라이브 코퍼스(8,981건)로 계산됐으나, loyalty_score 등 outcome 지표 4종은 실루엣 게이트를 아직 통과하지 못해 v7 40 시점 고정 스냅샷(7,350건)에 머물러 있다. v7 41~55 라운드에서 추가된 근거(v7 53· ...


,outcome(동결),r(MCI 최종),p,JSON r(MCI v7-55),r(MCI_excess 최종),JSON r(MCI_excess)
0,loyalty_score,-0.3849,0.0090,-0.3932,-0.0443,-0.0686
1,spillover_score,-0.0663,0.6651,-0.0776,0.2138,0.1888
2,coverage_index,-0.0560,0.7149,-0.0390,0.1275,0.1571
3,factor_diversity,-0.0622,0.6849,-0.0547,-0.1565,-0.1425


## 5. MCI 해석 구간(예시 임계값 0.30/0.45 — 문서에 정확한 컷오프 없음) + FPU JSON 스키마 예시

In [7]:
def classify_mci(m):
    return "분산형(Distributed)" if m < 0.30 else ("다극형(Multi-node)" if m < 0.45 else "스타중심형(Star-centered)")
member_df["mci_bucket_illustrative"] = member_df["mci_index"].apply(classify_mci)
print("⚠️ 버킷 경계값(0.30 / 0.45)은 예시 설정이다.")
print(member_df["mci_bucket_illustrative"].value_counts())

def build_fpu_example(group):
    rec = mm[group]; live = next((r for r in live_scores if r["fandom"] == group), None)
    return {"fpu_id": f"fpu_{group}", "schema_version": "v5.0_index",
            "group_core": {"name": group, "total_group_bullets": rec["total_group_bullets"], "coverage_index(live)": live["coverage_index"] if live else None},
            "unit_hierarchy": None, "unit_hierarchy_status": "not_yet_computable",
            "member_nodes": [{"node_type": "member", "name": m, "mentions": c, "impact_share": rec["member_impact_share_index"][m],
                              "activation_score": None, "activation_status": "not_yet_computable"} for m, c in rec["member_mention_counts"].items()],
            "mci_index": rec["mci_index"], "joint_evidence": None, "synergy_score": None, "dedup_applied": False, "source_note": rec["note"]}
print(json.dumps(build_fpu_example("빅뱅"), ensure_ascii=False, indent=2))

⚠️ 버킷 경계값(0.30 / 0.45)은 예시 설정이다.
mci_bucket_illustrative
분산형(Distributed)        36
다극형(Multi-node)          5
스타중심형(Star-centered)     4
Name: count, dtype: int64
{
  "fpu_id": "fpu_빅뱅",
  "schema_version": "v5.0_index",
  "group_core": {
    "name": "빅뱅",
    "total_group_bullets": 97,
    "coverage_index(live)": 0.757
  },
  "unit_hierarchy": null,
  "unit_hierarchy_status": "not_yet_computable",
  "member_nodes": [
    {
      "node_type": "member",
      "name": "지드래곤",
      "mentions": 15,
      "impact_share": 0.469,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    },
    {
      "node_type": "member",
      "name": "태양",
      "mentions": 11,
      "impact_share": 0.344,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    },
    {
      "node_type": "member",
      "name": "대성",
      "mentions": 5,
      "impact_share": 0.156,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    

## 6. 한계 — 아직 계산할 수 없는 것

1. 멤버별 독립 리서치가 아니라 그룹 단위 근거문장 안의 멤버명 언급 수 기반 텍스트마이닝 지수다(각 레코드 `note`).
2. Unit 계층(NCT 유닛 등), Joint Evidence 분리, Group–Member Synergy, Member Activation Score, 중복 집계 방지(event_id)는 여전히 미구현이다.
3. 상관분석 JSON은 v7-55 시점(8,981건) MCI × 동결 outcome이며, 4절의 최종 MCI 재계산값과는 소폭 다르다 — 결론(멤버 수 통제 시 설명력 소멸)은 유지되는지 4절 표로 확인한다.
4. MCI의 구조적 하한 1/n 때문에 멤버 수가 적은 그룹은 MCI가 높게 나온다 — `mci_excess`를 함께 본다.